In [1]:
# =========================
# ALL-IN-ONE TRAINING CELL
# =========================

# Install first if needed:
# !pip install torch torchvision pandas openpyxl pydicom scipy scikit-learn numpy

import os
import re
import glob
import math
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from scipy.ndimage import zoom
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")

# =========================================================
# 1) CONFIG: EDIT THESE PATHS/COLUMNS ONLY
# =========================================================
EXCEL_PATH = r"../LIDC-IDRI-133GB/Data/tcia-diagnosis-data-2012-04-20.xls"        # path to your xls/xlsx file
DICOM_ROOT = r"../LIDC-IDRI-133GB/Data/manifest-1600709154662/LIDC-IDRI"   # root folder containing all patient folders
OUTPUT_MODEL = "best_multimodal_model.pt"

PATIENT_ID_COL = "TCIA Patient ID"
LABEL_SOURCE_COL = "Diagnosis at the Patient Level\n0=Unknown\n1=benign or non-malignant disease\n2= malignant, primary lung cancer\n3 = malignant metastatic\n"
LABEL_COL = "label"   # binary label created from LABEL_SOURCE_COL

# Leave as None to auto-detect numeric feature columns from the Excel file
FEATURE_COLS = None

# Training settings
TARGET_SPACING = (2.0, 1.0, 1.0)     # faster than full 1mm, good baseline
TARGET_SHAPE = (96, 160, 160)        # good balance of speed/quality
BATCH_SIZE = 2
EPOCHS = 15
LR = 1e-4
WEIGHT_DECAY = 1e-5
NUM_WORKERS = 0
VAL_SIZE = 0.2
SEED = 42
USE_AMP = True                       # mixed precision if CUDA exists

# =========================================================
# 2) REPRODUCIBILITY
# =========================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================================================
# 3) HELPERS
# =========================================================
def normalize_patient_id(x):
    s = str(x).strip()
    s = s.replace("\\", "/")
    s = os.path.basename(s)
    s = re.sub(r"\.xml$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\.xlsx?$", "", s, flags=re.IGNORECASE)
    m = re.search(r"LIDC-IDRI-\d+", s, flags=re.IGNORECASE)
    if m:
        digits = re.search(r"\d+", m.group(0)).group(0)
        return f"LIDC-IDRI-{int(digits):04d}"
    if re.fullmatch(r"\d+", s):
        return f"LIDC-IDRI-{int(s):04d}"
    return s

def find_dicom_series_dirs(root_dir):
    """
    Recursively find directories containing DICOM files.
    We keep directories that contain at least one .dcm file,
    or files readable by pydicom with PixelData.
    """
    root_dir = Path(root_dir)
    series_dirs = []

    for cur_root, dirs, files in os.walk(root_dir):
        files = list(files)
        dcm_files = [f for f in files if f.lower().endswith(".dcm")]
        if dcm_files:
            series_dirs.append(Path(cur_root))
            continue

        # fallback: some DICOMs have no extension
        # sample only a few files to avoid slowdown
        candidate_files = [f for f in files if "." not in f][:5]
        has_dicom = False
        for f in candidate_files:
            fp = Path(cur_root) / f
            try:
                ds = pydicom.dcmread(str(fp), stop_before_pixels=False, force=True)
                if hasattr(ds, "PixelData"):
                    has_dicom = True
                    break
            except Exception:
                pass
        if has_dicom:
            series_dirs.append(Path(cur_root))

    return sorted(list(set(series_dirs)))

def build_patient_to_series_map(root_dir):
    """
    Map patient_id -> best DICOM series dir.
    Heuristic:
      - collect all discovered series dirs
      - infer patient id from path parts containing 'LIDC-IDRI-*'
      - if multiple series found, choose the one with most slices
    """
    series_dirs = find_dicom_series_dirs(root_dir)
    patient_map = {}

    def count_dicom_files(d):
        files = list(Path(d).glob("*.dcm"))
        if len(files) > 0:
            return len(files)
        # fallback count files without extension
        return len([x for x in Path(d).iterdir() if x.is_file()])

    for d in series_dirs:
        parts = [p for p in d.parts]
        patient_ids = [p for p in parts if p.startswith("LIDC-IDRI-")]
        if not patient_ids:
            # fallback: use nearest folder name that looks like patient id
            m = re.search(r"LIDC-IDRI-\d+", str(d))
            if m:
                patient_id = normalize_patient_id(m.group(0))
            else:
                continue
        else:
            patient_id = normalize_patient_id(patient_ids[-1])

        score = count_dicom_files(d)
        if patient_id not in patient_map or score > patient_map[patient_id][1]:
            patient_map[patient_id] = (str(d), score)

    patient_map = {k: v[0] for k, v in patient_map.items()}
    return patient_map

def load_dicom_series(dicom_dir):
    dicom_dir = Path(dicom_dir)

    files = list(dicom_dir.glob("*.dcm"))
    if not files:
        files = [f for f in dicom_dir.iterdir() if f.is_file()]

    slices = []
    for fp in files:
        try:
            ds = pydicom.dcmread(str(fp), force=True)
            if hasattr(ds, "PixelData"):
                slices.append(ds)
        except Exception:
            pass

    if len(slices) == 0:
        raise ValueError(f"No readable DICOM slices in {dicom_dir}")

    def sort_key(ds):
        if hasattr(ds, "ImagePositionPatient"):
            try:
                return float(ds.ImagePositionPatient[2])
            except Exception:
                pass
        if hasattr(ds, "InstanceNumber"):
            try:
                return int(ds.InstanceNumber)
            except Exception:
                pass
        return 0

    slices = sorted(slices, key=sort_key)

    volume = np.empty((len(slices), slices[0].Rows, slices[0].Columns), dtype=np.float32)


    for i, ds in enumerate(slices):


        arr = ds.pixel_array.astype(np.float32)


        slope = float(getattr(ds, 'RescaleSlope', 1.0))


        inter = float(getattr(ds, 'RescaleIntercept', 0.0))


        volume[i] = arr * slope + inter


        if hasattr(ds, '_pixel_array'): del ds._pixel_array


        if hasattr(ds, 'PixelData'): del ds.PixelData

    try:
        z_positions = [float(ds.ImagePositionPatient[2]) for ds in slices]
        z_spacing = abs(z_positions[1] - z_positions[0]) if len(z_positions) > 1 else float(getattr(slices[0], "SliceThickness", 1.0))
        if z_spacing == 0:
            z_spacing = float(getattr(slices[0], "SliceThickness", 1.0))
    except Exception:
        z_spacing = float(getattr(slices[0], "SliceThickness", 1.0))

    pixel_spacing = getattr(slices[0], "PixelSpacing", [1.0, 1.0])
    y_spacing = float(pixel_spacing[0])
    x_spacing = float(pixel_spacing[1])

    spacing = (z_spacing, y_spacing, x_spacing)
    return volume, spacing

def clip_and_normalize_hu(volume, hu_min=-1000, hu_max=400):
    volume = np.clip(volume, hu_min, hu_max)
    volume = (volume - hu_min) / (hu_max - hu_min + 1e-8)
    return volume.astype(np.float32)

def resample_volume(volume, current_spacing, target_spacing=(2.0, 1.0, 1.0)):
    factors = (
        current_spacing[0] / target_spacing[0],
        current_spacing[1] / target_spacing[1],
        current_spacing[2] / target_spacing[2],
    )
    return zoom(volume, factors, order=1)

def center_crop_or_pad(volume, target_shape=(96, 160, 160)):
    tz, ty, tx = target_shape
    z, y, x = volume.shape

    out = np.zeros(target_shape, dtype=volume.dtype)

    z0s = max((z - tz) // 2, 0)
    y0s = max((y - ty) // 2, 0)
    x0s = max((x - tx) // 2, 0)

    z1s = min(z0s + tz, z)
    y1s = min(y0s + ty, y)
    x1s = min(x0s + tx, x)

    cz, cy, cx = z1s - z0s, y1s - y0s, x1s - x0s

    z0d = max((tz - cz) // 2, 0)
    y0d = max((ty - cy) // 2, 0)
    x0d = max((tx - cx) // 2, 0)

    out[z0d:z0d+cz, y0d:y0d+cy, x0d:x0d+cx] = volume[z0s:z1s, y0s:y1s, x0s:x1s]
    return out

def preprocess_ct(dicom_dir, target_spacing=TARGET_SPACING, target_shape=TARGET_SHAPE):
    volume, spacing = load_dicom_series(dicom_dir)
    volume = clip_and_normalize_hu(volume)
    volume = resample_volume(volume, spacing, target_spacing)
    volume = center_crop_or_pad(volume, target_shape)
    return volume

# =========================================================
# 4) LOAD EXCEL + MATCH ALL DICOM FOLDERS
# =========================================================
print("\nScanning DICOM folders...")
patient_to_series = build_patient_to_series_map(DICOM_ROOT)
print("Discovered patient series:", len(patient_to_series))

print("\nReading Excel...")
if EXCEL_PATH.lower().endswith(".csv"):
    df = pd.read_csv(EXCEL_PATH)
else:
    df = pd.read_excel(EXCEL_PATH)

if PATIENT_ID_COL not in df.columns:
    raise ValueError(f"Excel must contain '{PATIENT_ID_COL}' column")
if LABEL_SOURCE_COL not in df.columns:
    raise ValueError(f"Excel must contain '{LABEL_SOURCE_COL}' column")

df[PATIENT_ID_COL] = df[PATIENT_ID_COL].astype(str).map(normalize_patient_id)
df[LABEL_COL] = df[LABEL_SOURCE_COL].map(lambda x: 0 if pd.isna(x) or int(x) in (0, 1) else 1)

df["dicom_dir"] = df[PATIENT_ID_COL].map(patient_to_series)

before_rows = len(df)
df = df.dropna(subset=["dicom_dir", LABEL_COL]).copy()
after_rows = len(df)

print(f"Rows in Excel: {before_rows}")
print(f"Matched rows with DICOM: {after_rows}")
print(f"Unmatched rows removed: {before_rows - after_rows}")

if after_rows < 10:
    print("Warning: very few matched rows. Check DICOM_ROOT and patient ID normalization.")

# Auto-detect numeric feature columns if not supplied
if FEATURE_COLS is None:
    excluded = {PATIENT_ID_COL, LABEL_SOURCE_COL, LABEL_COL, "dicom_dir"}
    numeric_cols = [c for c in df.columns if c not in excluded and pd.api.types.is_numeric_dtype(df[c])]
    FEATURE_COLS = [c for c in numeric_cols if df[c].notna().any()]

if len(FEATURE_COLS) == 0:
    print("\nNo numeric feature columns found. Using image-only model behavior via dummy feature.")
    df["dummy_feature"] = 0.0
    FEATURE_COLS = ["dummy_feature"]

# Keep only required columns and clean
needed = [PATIENT_ID_COL, "dicom_dir", LABEL_COL] + FEATURE_COLS
df = df[needed].copy()
for col in FEATURE_COLS:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Cast labels to int
df[LABEL_COL] = df[LABEL_COL].astype(int)

print("\nUsing feature columns:")
print(FEATURE_COLS)

print("\nClass distribution:")
print(df[LABEL_COL].value_counts().sort_index())

# =========================================================
# 5) SPLIT + SCALE TABULAR
# =========================================================
train_df, val_df = train_test_split(
    df,
    test_size=VAL_SIZE,
    random_state=SEED,
    stratify=df[LABEL_COL]
)

scaler = StandardScaler()
if len(FEATURE_COLS) > 0:
    feature_medians = train_df[FEATURE_COLS].median(numeric_only=True)
    train_df[FEATURE_COLS] = train_df[FEATURE_COLS].fillna(feature_medians)
    val_df[FEATURE_COLS] = val_df[FEATURE_COLS].fillna(feature_medians)
    train_df[FEATURE_COLS] = scaler.fit_transform(train_df[FEATURE_COLS])
    val_df[FEATURE_COLS] = scaler.transform(val_df[FEATURE_COLS])
else:
    train_df["dummy_feature"] = 0.0
    val_df["dummy_feature"] = 0.0
    FEATURE_COLS = ["dummy_feature"]

# =========================================================
# 6) DATASET
# =========================================================
class MultiModalCTDataset(Dataset):
    def __init__(self, df, feature_cols, training=False):
        self.df = df.reset_index(drop=True)
        self.feature_cols = feature_cols
        self.training = training

    def __len__(self):
        return len(self.df)

    def augment(self, vol):
        # vol: [Z, Y, X]
        if random.random() < 0.5:
            vol = np.flip(vol, axis=2).copy()
        if random.random() < 0.5:
            vol = np.flip(vol, axis=1).copy()
        return vol

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        vol = preprocess_ct(row["dicom_dir"], TARGET_SPACING, TARGET_SHAPE)
        if self.training:
            vol = self.augment(vol)

        vol = torch.tensor(vol, dtype=torch.float32).unsqueeze(0)  # [1,Z,Y,X]
        tab = torch.tensor(row[self.feature_cols].values.astype(np.float32), dtype=torch.float32)
        label = torch.tensor(float(row[LABEL_COL]), dtype=torch.float32)

        return {
            "image": vol,
            "tabular": tab,
            "label": label,
            "patient_id": row[PATIENT_ID_COL]
        }

train_ds = MultiModalCTDataset(train_df, FEATURE_COLS, training=True)
val_ds = MultiModalCTDataset(val_df, FEATURE_COLS, training=False)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

# =========================================================
# 7) MODEL: STRONGER 3D CNN + TABULAR FUSION
# =========================================================
class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _, _ = x.shape
        w = self.pool(x).view(b, c)
        w = self.fc(w).view(b, c, 1, 1, 1)
        return x * w

class ResidualBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(out_ch)
        self.se = SEBlock3D(out_ch)

        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm3d(out_ch)
            )
        else:
            self.shortcut = nn.Identity()

        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        out = self.act(out + identity)
        return out

class ImageEncoder3D(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv3d(1, 16, kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm3d(16),
            nn.ReLU(inplace=True),
        )
        self.layer1 = nn.Sequential(
            ResidualBlock3D(16, 32, stride=2),
            ResidualBlock3D(32, 32, stride=1),
        )
        self.layer2 = nn.Sequential(
            ResidualBlock3D(32, 64, stride=2),
            ResidualBlock3D(64, 64, stride=1),
        )
        self.layer3 = nn.Sequential(
            ResidualBlock3D(64, 128, stride=2),
            ResidualBlock3D(128, 128, stride=1),
        )
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25)
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.pool(x)
        x = self.head(x)
        return x

class TabularEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        hidden = max(64, in_dim * 2)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(hidden, out_dim),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)

class FusionModel(nn.Module):
    def __init__(self, num_tabular_features):
        super().__init__()
        self.image_encoder = ImageEncoder3D(out_dim=256)
        self.tab_encoder = TabularEncoder(num_tabular_features, out_dim=64)
        self.classifier = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, image, tabular):
        img_feat = self.image_encoder(image)
        tab_feat = self.tab_encoder(tabular)
        fused = torch.cat([img_feat, tab_feat], dim=1)
        return self.classifier(fused).squeeze(1)

model = FusionModel(len(FEATURE_COLS)).to(device)

# handle class imbalance
num_pos = (train_df[LABEL_COL] == 1).sum()
num_neg = (train_df[LABEL_COL] == 0).sum()
pos_weight_value = max(num_neg / max(num_pos, 1), 1.0)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=device))
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scaler_amp = torch.cuda.amp.GradScaler(enabled=(USE_AMP and device.type == "cuda"))

print("\nModel ready.")
print("Positive class weight:", pos_weight_value)

# =========================================================
# 8) TRAIN / EVAL
# =========================================================
def run_epoch(loader, training=True):
    if training:
        model.train()
    else:
        model.eval()

    losses = []
    all_labels = []
    all_probs = []

    for batch in loader:
        image = batch["image"].to(device, non_blocking=True)
        tabular = batch["tabular"].to(device, non_blocking=True)
        label = batch["label"].to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                logits = model(image, tabular)
                loss = criterion(logits, label)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler_amp.scale(loss).backward()
                scaler_amp.step(optimizer)
                scaler_amp.update()

        probs = torch.sigmoid(logits).detach().cpu().numpy()
        labels = label.detach().cpu().numpy()

        losses.extend([loss.item()] * len(labels))
        all_probs.extend(probs.tolist())
        all_labels.extend(labels.tolist())

    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels).astype(int)
    preds = (all_probs >= 0.5).astype(int)

    acc = accuracy_score(all_labels, preds)
    f1 = f1_score(all_labels, preds, zero_division=0)

    try:
        auc = roc_auc_score(all_labels, all_probs)
    except Exception:
        auc = float("nan")

    return {
        "loss": float(np.mean(losses)),
        "acc": acc,
        "f1": f1,
        "auc": auc,
        "labels": all_labels,
        "probs": all_probs,
        "preds": preds,
    }

best_score = -1.0
history = []

print("\nStarting training...\n")
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(train_loader, training=True)
    val_metrics = run_epoch(val_loader, training=False)

    score = val_metrics["auc"]
    if math.isnan(score):
        score = val_metrics["f1"]

    history.append({
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_acc": train_metrics["acc"],
        "train_f1": train_metrics["f1"],
        "train_auc": train_metrics["auc"],
        "val_loss": val_metrics["loss"],
        "val_acc": val_metrics["acc"],
        "val_f1": val_metrics["f1"],
        "val_auc": val_metrics["auc"],
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss {train_metrics['loss']:.4f} | Train Acc {train_metrics['acc']:.4f} | Train F1 {train_metrics['f1']:.4f} | Train AUC {train_metrics['auc']:.4f} || "
        f"Val Loss {val_metrics['loss']:.4f} | Val Acc {val_metrics['acc']:.4f} | Val F1 {val_metrics['f1']:.4f} | Val AUC {val_metrics['auc']:.4f}"
    )

    if score > best_score:
        best_score = score
        torch.save({
            "model_state_dict": model.state_dict(),
            "scaler_mean": scaler.mean_.tolist(),
            "scaler_scale": scaler.scale_.tolist(),
            "feature_cols": FEATURE_COLS,
            "target_spacing": TARGET_SPACING,
            "target_shape": TARGET_SHAPE,
            "patient_id_col": PATIENT_ID_COL,
            "label_col": LABEL_COL,
        }, OUTPUT_MODEL)
        print(f"Saved best model -> {OUTPUT_MODEL}")

# =========================================================
# 9) FINAL REPORT
# =========================================================
print("\nFinal validation report using the last epoch:")
final_metrics = run_epoch(val_loader, training=False)
print("Accuracy:", round(final_metrics["acc"], 4))
print("F1:", round(final_metrics["f1"], 4))
print("ROC-AUC:", round(final_metrics["auc"], 4) if not math.isnan(final_metrics["auc"]) else "nan")
print("\nClassification report:")
print(classification_report(final_metrics["labels"], final_metrics["preds"], digits=4, zero_division=0))

history_df = pd.DataFrame(history)
history_df.to_csv("training_history.csv", index=False)
print("\nSaved training history -> training_history.csv")

matched_df = df[[PATIENT_ID_COL, "dicom_dir", LABEL_COL] + FEATURE_COLS].copy()
matched_df.to_csv("matched_training_rows.csv", index=False)
print("Saved matched rows -> matched_training_rows.csv")
print("Done.")

Device: cuda

Scanning DICOM folders...
Discovered patient series: 948

Reading Excel...
Rows in Excel: 157
Matched rows with DICOM: 151
Unmatched rows removed: 6

Using feature columns:
['Diagnosis Method\n0 = unknown\n1 = review of radiological images to show 2 years of stable nodule\n2 = biopsy\n3 = surgical resection\n4 = progression or response', 'Nodule 1\nDiagnosis at the Nodule Level \n0=Unknown\n1=benign or non-malignant disease\n2= malignant, primary lung cancer\n3 = malignant metastatic)\n', 'Nodule 1\nDiagnosis Method at the Nodule Level\n0 = unknown\n1 = review of radiological images to show 2 years of stable nodule\n2 = biopsy\n3 = surgical resection\n4 = progression or response\n', 'Nodule 2\nDiagnosis at the Nodule Level \n0=Unknown\n1=benign or non-malignant disease\n2= malignant, primary lung cancer\n3 = malignant metastatic)\n', 'Nodule 2\nDiagnosis Method at the Nodule Level\n0 = unknown\n1 = review of radiological images to show 2 years of stable nodule\n2 = biopsy

In [2]:
# =========================
# MODEL TEST CELL
# =========================
# Loads the saved best_multimodal_model.pt, randomly samples N patients,
# runs predictions, and prints a detailed comparison vs ground truth.

import os
import re
import random
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from scipy.ndimage import zoom
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    classification_report, confusion_matrix
)
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

warnings.filterwarnings("ignore")

# =========================================================
# CONFIG  (edit these to match your training config)
# =========================================================
EXCEL_PATH   = r"../LIDC-IDRI-133GB/Data/tcia-diagnosis-data-2012-04-20.xls"
DICOM_ROOT   = r"../LIDC-IDRI-133GB/Data/manifest-1600709154662/LIDC-IDRI"
MODEL_PATH   = "best_multimodal_model.pt"          # saved by the training script

PATIENT_ID_COL   = "TCIA Patient ID"
LABEL_SOURCE_COL = (
    "Diagnosis at the Patient Level\n"
    "0=Unknown\n"
    "1=benign or non-malignant disease\n"
    "2= malignant, primary lung cancer\n"
    "3 = malignant metastatic\n"
)
LABEL_COL = "label"

# How many patients to randomly sample for the test
N_SAMPLES  = 20    # set to None to test ALL matched patients
SEED       = 99    # separate seed from training for a fair random draw
USE_AMP    = True

# =========================================================
# REPRODUCIBILITY
# =========================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# =========================================================
# HELPERS  (identical to training script)
# =========================================================
def normalize_patient_id(x):
    s = str(x).strip().replace("\\", "/")
    s = os.path.basename(s)
    s = re.sub(r"\.xml$", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\.xlsx?$", "", s, flags=re.IGNORECASE)
    m = re.search(r"LIDC-IDRI-\d+", s, flags=re.IGNORECASE)
    if m:
        digits = re.search(r"\d+", m.group(0)).group(0)
        return f"LIDC-IDRI-{int(digits):04d}"
    if re.fullmatch(r"\d+", s):
        return f"LIDC-IDRI-{int(s):04d}"
    return s

def find_dicom_series_dirs(root_dir):
    root_dir = Path(root_dir)
    series_dirs = []
    for cur_root, dirs, files in os.walk(root_dir):
        dcm_files = [f for f in files if f.lower().endswith(".dcm")]
        if dcm_files:
            series_dirs.append(Path(cur_root))
            continue
        candidate_files = [f for f in files if "." not in f][:5]
        for f in candidate_files:
            try:
                ds = pydicom.dcmread(str(Path(cur_root) / f), stop_before_pixels=False, force=True)
                if hasattr(ds, "PixelData"):
                    series_dirs.append(Path(cur_root))
                    break
            except Exception:
                pass
    return sorted(list(set(series_dirs)))

def build_patient_to_series_map(root_dir):
    series_dirs = find_dicom_series_dirs(root_dir)
    patient_map = {}

    def count_dicom_files(d):
        files = list(Path(d).glob("*.dcm"))
        return len(files) if files else len([x for x in Path(d).iterdir() if x.is_file()])

    for d in series_dirs:
        patient_ids = [p for p in d.parts if p.startswith("LIDC-IDRI-")]
        if not patient_ids:
            m = re.search(r"LIDC-IDRI-\d+", str(d))
            if m:
                patient_id = normalize_patient_id(m.group(0))
            else:
                continue
        else:
            patient_id = normalize_patient_id(patient_ids[-1])
        score = count_dicom_files(d)
        if patient_id not in patient_map or score > patient_map[patient_id][1]:
            patient_map[patient_id] = (str(d), score)

    return {k: v[0] for k, v in patient_map.items()}

def load_dicom_series(dicom_dir):
    dicom_dir = Path(dicom_dir)
    files = list(dicom_dir.glob("*.dcm")) or [f for f in dicom_dir.iterdir() if f.is_file()]
    slices = []
    for fp in files:
        try:
            ds = pydicom.dcmread(str(fp), force=True)
            if hasattr(ds, "PixelData"):
                slices.append(ds)
        except Exception:
            pass
    if not slices:
        raise ValueError(f"No readable DICOM slices in {dicom_dir}")

    def sort_key(ds):
        try:
            return float(ds.ImagePositionPatient[2])
        except Exception:
            try:
                return int(ds.InstanceNumber)
            except Exception:
                return 0

    slices = sorted(slices, key=sort_key)
    volume = np.empty((len(slices), slices[0].Rows, slices[0].Columns), dtype=np.float32)

    for i, ds in enumerate(slices):

        arr = ds.pixel_array.astype(np.float32)

        slope = float(getattr(ds, 'RescaleSlope', 1.0))

        inter = float(getattr(ds, 'RescaleIntercept', 0.0))

        volume[i] = arr * slope + inter

        if hasattr(ds, '_pixel_array'): del ds._pixel_array

        if hasattr(ds, 'PixelData'): del ds.PixelData

    try:
        z_pos    = [float(ds.ImagePositionPatient[2]) for ds in slices]
        z_space  = abs(z_pos[1] - z_pos[0]) if len(z_pos) > 1 else float(getattr(slices[0], "SliceThickness", 1.0))
        z_space  = z_space or float(getattr(slices[0], "SliceThickness", 1.0))
    except Exception:
        z_space  = float(getattr(slices[0], "SliceThickness", 1.0))

    ps = getattr(slices[0], "PixelSpacing", [1.0, 1.0])
    return volume, (z_space, float(ps[0]), float(ps[1]))

def clip_and_normalize_hu(volume, hu_min=-1000, hu_max=400):
    volume = np.clip(volume, hu_min, hu_max)
    return ((volume - hu_min) / (hu_max - hu_min + 1e-8)).astype(np.float32)

def resample_volume(volume, current_spacing, target_spacing):
    factors = tuple(c / t for c, t in zip(current_spacing, target_spacing))
    return zoom(volume, factors, order=1)

def center_crop_or_pad(volume, target_shape):
    tz, ty, tx = target_shape
    z,  y,  x  = volume.shape
    out  = np.zeros(target_shape, dtype=volume.dtype)
    z0s  = max((z - tz) // 2, 0);  y0s = max((y - ty) // 2, 0);  x0s = max((x - tx) // 2, 0)
    z1s  = min(z0s + tz, z);        y1s = min(y0s + ty, y);        x1s = min(x0s + tx, x)
    cz, cy, cx = z1s - z0s, y1s - y0s, x1s - x0s
    z0d = max((tz - cz) // 2, 0);  y0d = max((ty - cy) // 2, 0);  x0d = max((tx - cx) // 2, 0)
    out[z0d:z0d+cz, y0d:y0d+cy, x0d:x0d+cx] = volume[z0s:z1s, y0s:y1s, x0s:x1s]
    return out

def preprocess_ct(dicom_dir, target_spacing, target_shape):
    volume, spacing = load_dicom_series(dicom_dir)
    volume = clip_and_normalize_hu(volume)
    volume = resample_volume(volume, spacing, target_spacing)
    volume = center_crop_or_pad(volume, target_shape)
    return volume

# =========================================================
# MODEL DEFINITION  (must match training script exactly)
# =========================================================
class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )
    def forward(self, x):
        b, c = x.shape[:2]
        w = self.pool(x).view(b, c)
        return x * self.fc(w).view(b, c, 1, 1, 1)

class ResidualBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1    = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1      = nn.BatchNorm3d(out_ch)
        self.conv2    = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2      = nn.BatchNorm3d(out_ch)
        self.se       = SEBlock3D(out_ch)
        self.shortcut = (
            nn.Sequential(nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False), nn.BatchNorm3d(out_ch))
            if stride != 1 or in_ch != out_ch else nn.Identity()
        )
        self.act = nn.ReLU(inplace=True)
    def forward(self, x):
        identity = self.shortcut(x)
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return self.act(out + identity)

class ImageEncoder3D(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.stem   = nn.Sequential(nn.Conv3d(1, 16, 5, stride=2, padding=2, bias=False), nn.BatchNorm3d(16), nn.ReLU(inplace=True))
        self.layer1 = nn.Sequential(ResidualBlock3D(16, 32, 2), ResidualBlock3D(32, 32))
        self.layer2 = nn.Sequential(ResidualBlock3D(32, 64, 2), ResidualBlock3D(64, 64))
        self.layer3 = nn.Sequential(ResidualBlock3D(64, 128, 2), ResidualBlock3D(128, 128))
        self.pool   = nn.AdaptiveAvgPool3d(1)
        self.head   = nn.Sequential(nn.Flatten(), nn.Linear(128, out_dim), nn.ReLU(inplace=True), nn.Dropout(0.25))
    def forward(self, x):
        return self.head(self.pool(self.layer3(self.layer2(self.layer1(self.stem(x))))))

class TabularEncoder(nn.Module):
    def __init__(self, in_dim, out_dim=64):
        super().__init__()
        hidden  = max(64, in_dim * 2)
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.BatchNorm1d(hidden), nn.ReLU(inplace=True),
            nn.Dropout(0.2), nn.Linear(hidden, out_dim), nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.net(x)

class FusionModel(nn.Module):
    def __init__(self, num_tabular_features):
        super().__init__()
        self.image_encoder = ImageEncoder3D(out_dim=256)
        self.tab_encoder   = TabularEncoder(num_tabular_features, out_dim=64)
        self.classifier    = nn.Sequential(
            nn.Linear(320, 128), nn.ReLU(inplace=True), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.ReLU(inplace=True), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, image, tabular):
        fused = torch.cat([self.image_encoder(image), self.tab_encoder(tabular)], dim=1)
        return self.classifier(fused).squeeze(1)

# =========================================================
# LOAD CHECKPOINT
# =========================================================
print(f"\nLoading model checkpoint from: {MODEL_PATH}")
ckpt = torch.load(MODEL_PATH, map_location=device)

feature_cols    = ckpt["feature_cols"]
target_spacing  = tuple(ckpt["target_spacing"])
target_shape    = tuple(ckpt["target_shape"])

scaler = StandardScaler()
scaler.mean_  = np.array(ckpt["scaler_mean"])
scaler.scale_ = np.array(ckpt["scaler_scale"])

model = FusionModel(len(feature_cols)).to(device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Model loaded. Feature cols: {feature_cols}")
print(f"Target spacing: {target_spacing}  |  Target shape: {target_shape}")

# =========================================================
# BUILD DATA TABLE
# =========================================================
print("\nScanning DICOM folders...")
patient_to_series = build_patient_to_series_map(DICOM_ROOT)
print(f"Discovered patients: {len(patient_to_series)}")

print("Reading Excel...")
df = pd.read_csv(EXCEL_PATH) if EXCEL_PATH.lower().endswith(".csv") else pd.read_excel(EXCEL_PATH)
df[PATIENT_ID_COL] = df[PATIENT_ID_COL].astype(str).map(normalize_patient_id)
df[LABEL_COL]      = df[LABEL_SOURCE_COL].map(lambda x: 0 if pd.isna(x) or int(x) in (0, 1) else 1)
df["dicom_dir"]    = df[PATIENT_ID_COL].map(patient_to_series)
df = df.dropna(subset=["dicom_dir", LABEL_COL]).copy()
df[LABEL_COL] = df[LABEL_COL].astype(int)

# Fill and scale tabular features using the checkpoint's scaler
feature_medians = df[feature_cols].median(numeric_only=True)
df[feature_cols] = df[feature_cols].fillna(feature_medians)
df[feature_cols] = scaler.transform(df[feature_cols])

print(f"Total matched patients available: {len(df)}")

# =========================================================
# RANDOM SAMPLING
# =========================================================
n = min(N_SAMPLES, len(df)) if N_SAMPLES else len(df)
sample_df = df.sample(n=n, random_state=SEED).reset_index(drop=True)
print(f"\nRandomly selected {n} patients for testing (seed={SEED})\n")

# =========================================================
# INFERENCE LOOP
# =========================================================
LABEL_NAMES = {0: "Benign/Unknown", 1: "Malignant"}

results = []
print(f"{'#':<4} {'Patient ID':<20} {'Actual':<20} {'Predicted':<20} {'Probability':>11}  {'Correct?'}")
print("-" * 85)

with torch.no_grad():
    for i, row in sample_df.iterrows():
        patient_id = row[PATIENT_ID_COL]
        actual     = int(row[LABEL_COL])

        try:
            vol = preprocess_ct(row["dicom_dir"], target_spacing, target_shape)
            image   = torch.tensor(vol, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)
            tabular = torch.tensor(
                row[feature_cols].values.astype(np.float32), dtype=torch.float32
            ).unsqueeze(0).to(device)

            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                logit = model(image, tabular)

            prob      = torch.sigmoid(logit).item()
            predicted = int(prob >= 0.5)
            correct   = "✓" if predicted == actual else "✗"
            error_msg = None

        except Exception as e:
            prob = float("nan")
            predicted = -1
            correct   = "ERR"
            error_msg = str(e)

        results.append({
            "patient_id": patient_id,
            "actual":     actual,
            "actual_label": LABEL_NAMES.get(actual, "?"),
            "predicted":  predicted,
            "predicted_label": LABEL_NAMES.get(predicted, "Error"),
            "probability": prob,
            "correct":    correct == "✓",
            "error":      error_msg,
        })

        row_num = len(results)
        print(
            f"{row_num:<4} {patient_id:<20} "
            f"{LABEL_NAMES.get(actual,'?'):<20} "
            f"{LABEL_NAMES.get(predicted,'Error'):<20} "
            f"{prob:>11.4f}  {correct}"
        )

# =========================================================
# AGGREGATE METRICS
# =========================================================
results_df = pd.DataFrame(results)
valid      = results_df[results_df["predicted"] != -1].copy()

print("\n" + "=" * 85)
print("SUMMARY")
print("=" * 85)
print(f"Total samples tested : {len(results_df)}")
print(f"Successful inferences: {len(valid)}")
print(f"Errors               : {len(results_df) - len(valid)}")

if len(valid) > 0:
    y_true  = valid["actual"].values.astype(int)
    y_pred  = valid["predicted"].values.astype(int)
    y_prob  = valid["probability"].values

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_prob)
    except Exception:
        auc = float("nan")

    print(f"\nAccuracy : {acc:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"ROC-AUC  : {auc:.4f}" if not math.isnan(auc) else "ROC-AUC  : N/A (only one class present in sample)")

    print("\nConfusion Matrix (rows=Actual, cols=Predicted):")
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    print(f"               Pred Benign  Pred Malignant")
    print(f"Actual Benign      {cm[0,0]:<11} {cm[0,1]}")
    print(f"Actual Malignant   {cm[1,0]:<11} {cm[1,1]}")

    print("\nDetailed Classification Report:")
    print(classification_report(y_true, y_pred, target_names=["Benign/Unknown", "Malignant"],
                                digits=4, zero_division=0))

    print("\nPer-patient breakdown:")
    print(results_df[[
        "patient_id", "actual_label", "predicted_label", "probability", "correct"
    ]].to_string(index=False))

# =========================================================
# SAVE RESULTS
# =========================================================
out_csv = "test_predictions.csv"
results_df.to_csv(out_csv, index=False)
print(f"\nResults saved to: {out_csv}")

# Report any errors encountered
errors = results_df[results_df["error"].notna()]
if len(errors) > 0:
    print(f"\n  {len(errors)} patient(s) failed during inference:")
    for _, r in errors.iterrows():
        print(f"  {r['patient_id']}: {r['error']}")

Device: cuda

Loading model checkpoint from: best_multimodal_model.pt
Model loaded. Feature cols: ['Diagnosis Method\n0 = unknown\n1 = review of radiological images to show 2 years of stable nodule\n2 = biopsy\n3 = surgical resection\n4 = progression or response', 'Nodule 1\nDiagnosis at the Nodule Level \n0=Unknown\n1=benign or non-malignant disease\n2= malignant, primary lung cancer\n3 = malignant metastatic)\n', 'Nodule 1\nDiagnosis Method at the Nodule Level\n0 = unknown\n1 = review of radiological images to show 2 years of stable nodule\n2 = biopsy\n3 = surgical resection\n4 = progression or response\n', 'Nodule 2\nDiagnosis at the Nodule Level \n0=Unknown\n1=benign or non-malignant disease\n2= malignant, primary lung cancer\n3 = malignant metastatic)\n', 'Nodule 2\nDiagnosis Method at the Nodule Level\n0 = unknown\n1 = review of radiological images to show 2 years of stable nodule\n2 = biopsy\n3 = surgical resection\n4 = progression or response\n']
Target spacing: (2.0, 1.0, 1.0)

------------------------------***********--------------------------------------------

In [3]:
# ==============================================================================
# LIDC-IDRI  |  Multiclass Lung Cancer Classification  |  Research-Grade Pipeline
# ==============================================================================
#
# Classes:
#   0 = Benign / Non-malignant / Unknown
#   1 = Malignant – Primary Lung Cancer
#   2 = Malignant – Metastatic
#
# Architecture:
#   - 3D ResNet-SE backbone with CBAM attention
#   - Image-only model (no tabular features — see leakage analysis below)
#   - Focal loss + class weighting for imbalance
#   - Stratified K-Fold cross-validation
#   - Mixed precision (AMP), early stopping, cosine LR scheduler
#   - Comprehensive evaluation: accuracy, macro/weighted F1, ROC-AUC (OvR),
#     per-class precision/recall, confusion matrix, calibration curve
#
# ── LEAKAGE ANALYSIS ─────────────────────────────────────────────────────────
# The TCIA Excel sheet (tcia-diagnosis-data-2012-04-20.xls) contains 14 columns:
#   Col 0 : TCIA Patient ID              ← identifier only, safe
#   Col 1 : Patient-level diagnosis      ← THE LABEL SOURCE — never a feature
#   Col 2 : Diagnosis Method             ← LEAKAGE (derived from diagnostic process)
#   Col 3 : Primary tumor site           ← LEAKAGE (only exists when malignant)
#   Col 4–13: Nodule-level diagnoses &   ← LEAKAGE (direct label derivatives)
#             nodule-level methods
# Conclusion: Zero safe tabular features exist in this spreadsheet.
#             The model operates in image-only mode.
# ─────────────────────────────────────────────────────────────────────────────
#
# Install (if needed):
#   pip install torch torchvision pandas openpyxl xlrd pydicom scipy
#              scikit-learn numpy matplotlib seaborn
# ==============================================================================

import os
import re
import math
import glob
import json
import time
import random
import warnings
import logging
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.ndimage import zoom, rotate, gaussian_filter, map_coordinates

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix,
    precision_score, recall_score
)
from sklearn.calibration import calibration_curve

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(asctime)s  %(message)s")
log = logging.getLogger(__name__)

# ==============================================================================
# 1.  CONFIGURATION
# ==============================================================================
class Config:
    # ── Paths ──────────────────────────────────────────────────────────────
    EXCEL_PATH   = r"../LIDC-IDRI-133GB/Data/tcia-diagnosis-data-2012-04-20.xls"
    DICOM_ROOT   = r"../LIDC-IDRI-133GB/Data/manifest-1600709154662/LIDC-IDRI"
    OUTPUT_DIR   = "lidc_output"

    # ── Label column in the Excel file ─────────────────────────────────────
    PATIENT_ID_COL   = "TCIA Patient ID"
    LABEL_SOURCE_COL = (
        "Diagnosis at the Patient Level\n"
        "0=Unknown\n"
        "1=benign or non-malignant disease\n"
        "2= malignant, primary lung cancer\n"
        "3 = malignant metastatic\n"
    )
    # Mapping: Excel value → class index
    # Excel 0 (unknown) → merged into class 0 (conservative: treat as benign)
    # Excel 1 (benign)  → class 0
    # Excel 2 (primary) → class 1
    # Excel 3 (metast.) → class 2
    LABEL_MAP = {0: 0, 1: 0, 2: 1, 3: 2}
    NUM_CLASSES = 3
    CLASS_NAMES = ["Benign/Unknown", "Primary Lung Ca", "Metastatic"]

    # ── CT Preprocessing ───────────────────────────────────────────────────
    HU_MIN            = -1000
    HU_MAX            =  400
    TARGET_SPACING    = (2.0, 1.0, 1.0)   # mm  (z, y, x)
    TARGET_SHAPE      = (64, 128, 128)     # voxels  — increase if GPU RAM allows
    # Lung window for a second channel (soft-tissue contrast)
    HU_LUNG_MIN       = -600
    HU_LUNG_MAX       =  1500
    USE_DUAL_CHANNEL  = True               # channel 0=standard, channel 1=lung-window
    IN_CHANNELS       = 2 if USE_DUAL_CHANNEL else 1

    # ── Augmentation ───────────────────────────────────────────────────────
    AUG_FLIP_PROB          = 0.5
    AUG_ROTATE_PROB        = 0.5
    AUG_ROTATE_MAX_DEG     = 15.0
    AUG_GAUSSIAN_NOISE_STD = 0.02
    AUG_NOISE_PROB         = 0.3
    AUG_GAMMA_PROB         = 0.3
    AUG_GAMMA_RANGE        = (0.75, 1.25)
    AUG_ELASTIC_PROB       = 0.2
    AUG_ELASTIC_ALPHA      = 10.0
    AUG_ELASTIC_SIGMA      = 2.0

    # ── Model ──────────────────────────────────────────────────────────────
    BACKBONE_CHANNELS = [16, 32, 64, 128, 256]   # stem + 4 stages
    EMBED_DIM         = 256                        # image embedding size

    # ── Training ───────────────────────────────────────────────────────────
    BATCH_SIZE    = 2
    EPOCHS        = 40
    LR            = 3e-4
    WEIGHT_DECAY  = 1e-4
    NUM_WORKERS   = 0
    USE_AMP       = True
    SEED          = 42

    # K-Fold
    N_FOLDS       = 5     # set to 1 to skip K-Fold and use a single 80/20 split

    # Early stopping
    PATIENCE      = 8

    # LR scheduler: cosine annealing
    LR_MIN        = 1e-6

    # Focal Loss
    USE_FOCAL     = True
    FOCAL_GAMMA   = 2.0

    # Validation split (only used when N_FOLDS == 1)
    VAL_SIZE      = 0.20

    # Threshold tuning: optimize per-class thresholds on validation set
    TUNE_THRESHOLDS = True

    # ── Output ─────────────────────────────────────────────────────────────
    MODEL_NAME    = "best_lidc_model"

C = Config()

Path(C.OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

# ==============================================================================
# 2.  REPRODUCIBILITY
# ==============================================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed(C.SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
log.info(f"Device: {device}")

# ==============================================================================
# 3.  DICOM DISCOVERY & LOADING
# ==============================================================================
def normalize_patient_id(x: str) -> str:
    s = str(x).strip().replace("\\", "/")
    s = os.path.basename(s)
    s = re.sub(r"\.(xml|xlsx?)$", "", s, flags=re.IGNORECASE)
    m = re.search(r"LIDC-IDRI-\d+", s, flags=re.IGNORECASE)
    if m:
        digits = re.search(r'\d+', m.group(0)).group(0)
        return f"LIDC-IDRI-{int(digits):04d}"
    if re.fullmatch(r"\d+", s):
        return f"LIDC-IDRI-{int(s):04d}"
    return s

def find_dicom_series_dirs(root_dir):
    root_dir = Path(root_dir)
    series_dirs = []

    for cur_root, dirs, files in os.walk(str(root_dir)):
        dcm_files = [f for f in files if f.lower().endswith(".dcm")]
        if dcm_files:
            series_dirs.append(Path(cur_root))
            continue

        candidate_files = [f for f in files if "." not in f][:5]
        for f in candidate_files:
            fp = Path(cur_root) / f
            try:
                ds = pydicom.dcmread(str(fp), stop_before_pixels=False, force=True)
                if hasattr(ds, "PixelData"):
                    series_dirs.append(Path(cur_root))
                    break
            except Exception:
                pass

    return sorted(list(set(series_dirs)))


def build_patient_to_series_map(root_dir: str) -> dict:
    log.info("Scanning DICOM directory tree …")
    series_dirs = find_dicom_series_dirs(root_dir)
    log.info(f"  Total series directories found: {len(series_dirs)}")
    patient_map = {}

    def count_dicom_files(d):
        files = list(Path(d).glob("*.dcm"))
        return len(files) if files else len([x for x in Path(d).iterdir() if x.is_file()])

    for d in series_dirs:
        patient_ids = [
            p for p in d.parts
            if re.fullmatch(r"LIDC-IDRI-\d{4}", p, re.IGNORECASE)
        ]

        if not patient_ids:
            continue

        raw_id     = patient_ids[-1]
        digits     = re.search(r"\d+", raw_id).group(0)
        patient_id = f"LIDC-IDRI-{int(digits):04d}"

        score = count_dicom_files(d)
        if patient_id not in patient_map or score > patient_map[patient_id][1]:
            patient_map[patient_id] = (str(d), score)

    log.info(f"  Mapped {len(patient_map)} unique patients.")
    return {k: v[0] for k, v in patient_map.items()}

def load_dicom_series(dicom_dir: str):
    """
    Load a DICOM series → (volume [Z,Y,X] float32, spacing tuple (z,y,x)).
    """
    dicom_dir = Path(dicom_dir)
    files     = sorted(dicom_dir.glob("*.dcm")) or sorted(dicom_dir.iterdir())

    slices = []
    for fp in files:
        try:
            ds = pydicom.dcmread(str(fp), force=True)
            if hasattr(ds, "PixelData"):
                slices.append(ds)
        except Exception:
            pass

    if not slices:
        raise ValueError(f"No readable DICOM slices in {dicom_dir}")

    def _sort_key(ds):
        try:    return float(ds.ImagePositionPatient[2])
        except Exception: pass
        try:    return int(ds.InstanceNumber)
        except Exception: return 0

    slices.sort(key=_sort_key)

    volume = np.empty((len(slices), slices[0].Rows, slices[0].Columns), dtype=np.float32)


    for i, ds in enumerate(slices):


        arr = ds.pixel_array.astype(np.float32)


        slope = float(getattr(ds, 'RescaleSlope', 1.0))


        inter = float(getattr(ds, 'RescaleIntercept', 0.0))


        volume[i] = arr * slope + inter


        if hasattr(ds, '_pixel_array'): del ds._pixel_array


        if hasattr(ds, 'PixelData'): del ds.PixelData

    # spacing
    try:
        zs = [float(ds.ImagePositionPatient[2]) for ds in slices]
        z_sp = abs(zs[1] - zs[0]) if len(zs) > 1 else float(getattr(slices[0], "SliceThickness", 2.0))
        z_sp = z_sp or float(getattr(slices[0], "SliceThickness", 2.0))
    except Exception:
        z_sp = float(getattr(slices[0], "SliceThickness", 2.0))

    ps   = getattr(slices[0], "PixelSpacing", [1.0, 1.0])
    return volume, (z_sp, float(ps[0]), float(ps[1]))

# ==============================================================================
# 4.  CT PREPROCESSING
# ==============================================================================
def _clip_norm(volume: np.ndarray, hu_min: float, hu_max: float) -> np.ndarray:
    v = np.clip(volume, hu_min, hu_max)
    return ((v - hu_min) / (hu_max - hu_min + 1e-8)).astype(np.float32)


def _resample(volume: np.ndarray, spacing, target_spacing) -> np.ndarray:
    factors = tuple(s / t for s, t in zip(spacing, target_spacing))
    import torch
    import torch.nn.functional as F
    new_shape = (
        int(round(volume.shape[0] * factors[0])),
        int(round(volume.shape[1] * factors[1])),
        int(round(volume.shape[2] * factors[2]))
    )
    vol_t = torch.from_numpy(volume).unsqueeze(0).unsqueeze(0)
    vol_t = F.interpolate(vol_t, size=new_shape, mode='trilinear', align_corners=False)
    return vol_t.squeeze(0).squeeze(0).numpy()


def _crop_pad(volume: np.ndarray, shape) -> np.ndarray:
    tz, ty, tx = shape
    z,  y,  x  = volume.shape
    out = np.zeros(shape, dtype=volume.dtype)

    z0s = max((z - tz) // 2, 0);  y0s = max((y - ty) // 2, 0);  x0s = max((x - tx) // 2, 0)
    z1s = min(z0s + tz, z);        y1s = min(y0s + ty, y);        x1s = min(x0s + tx, x)
    cz, cy, cx = z1s - z0s, y1s - y0s, x1s - x0s

    z0d = max((tz - cz) // 2, 0);  y0d = max((ty - cy) // 2, 0);  x0d = max((tx - cx) // 2, 0)
    out[z0d:z0d+cz, y0d:y0d+cy, x0d:x0d+cx] = volume[z0s:z1s, y0s:y1s, x0s:x1s]
    return out


def preprocess_ct(dicom_dir: str) -> np.ndarray:
    """
    Returns numpy array of shape [C, Z, Y, X] (C=1 or 2).
    """
    volume, spacing = load_dicom_series(dicom_dir)
    volume          = _resample(volume, spacing, C.TARGET_SPACING)
    volume          = _crop_pad(volume, C.TARGET_SHAPE)

    ch_std  = _clip_norm(volume, C.HU_MIN,      C.HU_MAX)
    if C.USE_DUAL_CHANNEL:
        ch_lung = _clip_norm(volume, C.HU_LUNG_MIN, C.HU_LUNG_MAX)
        return np.stack([ch_std, ch_lung], axis=0)   # [2, Z, Y, X]
    return ch_std[np.newaxis]                          # [1, Z, Y, X]

# ==============================================================================
# 5.  AUGMENTATION
# ==============================================================================
def elastic_deform_3d(volume: np.ndarray, alpha: float, sigma: float) -> np.ndarray:
    """Apply random elastic deformation to a [C,Z,Y,X] volume."""
    C_, Z, Y, X = volume.shape
    # generate displacement fields on spatial axes only
    dx = gaussian_filter(np.random.randn(Z, Y, X).astype(np.float32), sigma) * alpha
    dy = gaussian_filter(np.random.randn(Z, Y, X).astype(np.float32), sigma) * alpha
    dz = gaussian_filter(np.random.randn(Z, Y, X).astype(np.float32), sigma) * alpha

    zz, yy, xx = np.meshgrid(
        np.arange(Z), np.arange(Y), np.arange(X), indexing="ij"
    )
    coords_z = np.clip(zz + dz, 0, Z - 1).ravel()
    coords_y = np.clip(yy + dy, 0, Y - 1).ravel()
    coords_x = np.clip(xx + dx, 0, X - 1).ravel()

    out = np.zeros_like(volume)
    for c in range(C_):
        out[c] = map_coordinates(volume[c], [coords_z, coords_y, coords_x],
                                 order=1, mode="reflect").reshape(Z, Y, X)
    return out


def augment(volume: np.ndarray) -> np.ndarray:
    """
    volume: [C, Z, Y, X] float32, values in [0, 1].
    Returns augmented copy.
    """
    v = volume.copy()

    # Random flips
    if random.random() < C.AUG_FLIP_PROB:
        v = np.flip(v, axis=3).copy()   # L/R
    if random.random() < C.AUG_FLIP_PROB:
        v = np.flip(v, axis=2).copy()   # A/P
    if random.random() < C.AUG_FLIP_PROB:
        v = np.flip(v, axis=1).copy()   # S/I (axial)

    # Random rotation around z-axis
    if random.random() < C.AUG_ROTATE_PROB:
        angle = random.uniform(-C.AUG_ROTATE_MAX_DEG, C.AUG_ROTATE_MAX_DEG)
        for c in range(v.shape[0]):
            v[c] = rotate(v[c], angle, axes=(1, 2), reshape=False, order=1, mode="reflect")

    # Gaussian noise
    if random.random() < C.AUG_NOISE_PROB:
        noise = np.random.randn(*v.shape).astype(np.float32) * C.AUG_GAUSSIAN_NOISE_STD
        v     = np.clip(v + noise, 0.0, 1.0)

    # Gamma / contrast shift (per channel)
    if random.random() < C.AUG_GAMMA_PROB:
        gamma = random.uniform(*C.AUG_GAMMA_RANGE)
        v     = np.clip(v ** gamma, 0.0, 1.0)

    # Elastic deformation
    if random.random() < C.AUG_ELASTIC_PROB:
        v = elastic_deform_3d(v, C.AUG_ELASTIC_ALPHA, C.AUG_ELASTIC_SIGMA)
        v = np.clip(v, 0.0, 1.0)

    return v.astype(np.float32)

# ==============================================================================
# 6.  DATASET
# ==============================================================================
class LIDCDataset(Dataset):
    def __init__(self, df: pd.DataFrame, training: bool = False):
        self.df       = df.reset_index(drop=True)
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row["label"])

        try:
            vol = preprocess_ct(row["dicom_dir"])
        except Exception as e:
            log.warning(f"Failed to load {row['dicom_dir']}: {e} — using zeros")
            vol = np.zeros((C.IN_CHANNELS, *C.TARGET_SHAPE), dtype=np.float32)

        if self.training:
            vol = augment(vol)

        return {
            "image":      torch.tensor(vol,   dtype=torch.float32),
            "label":      torch.tensor(label, dtype=torch.long),
            "patient_id": row["patient_id"],
        }

# ==============================================================================
# 7.  MODEL ARCHITECTURE
# ==============================================================================

# ── SE Block ─────────────────────────────────────────────────────────────────
class SEBlock3D(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        hidden    = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc   = nn.Sequential(
            nn.Linear(channels, hidden), nn.ReLU(inplace=True),
            nn.Linear(hidden, channels), nn.Sigmoid()
        )

    def forward(self, x):
        b, c = x.shape[:2]
        w    = self.pool(x).view(b, c)
        return x * self.fc(w).view(b, c, 1, 1, 1)


# ── CBAM Spatial Attention ───────────────────────────────────────────────────
class CBAMSpatial3D(nn.Module):
    def __init__(self, kernel_size: int = 7):
        super().__init__()
        pad       = kernel_size // 2
        self.conv = nn.Conv3d(2, 1, kernel_size, padding=pad, bias=False)

    def forward(self, x):
        avg_p = x.mean(dim=1, keepdim=True)
        max_p = x.max(dim=1, keepdim=True).values
        w     = torch.sigmoid(self.conv(torch.cat([avg_p, max_p], dim=1)))
        return x * w


# ── CBAM = SE (channel) + Spatial ────────────────────────────────────────────
class CBAM3D(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.se      = SEBlock3D(channels)
        self.spatial = CBAMSpatial3D()

    def forward(self, x):
        return self.spatial(self.se(x))


# ── Residual Block ────────────────────────────────────────────────────────────
class ResBlock3D(nn.Module):
    def __init__(self, in_ch: int, out_ch: int, stride: int = 1, use_cbam: bool = True):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm3d(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm3d(out_ch)
        self.attn  = CBAM3D(out_ch) if use_cbam else nn.Identity()
        self.down  = (
            nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm3d(out_ch)
            )
            if stride != 1 or in_ch != out_ch else nn.Identity()
        )
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        out = self.act(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.attn(out)
        return self.act(out + self.down(x))


# ── Main Encoder ──────────────────────────────────────────────────────────────
class Encoder3D(nn.Module):
    """
    5-stage 3D ResNet with CBAM, producing a flat embed of `out_dim` features.
    """
    def __init__(self, in_channels: int = 1, out_dim: int = 256):
        super().__init__()
        ch = C.BACKBONE_CHANNELS   # [16, 32, 64, 128, 256]

        self.stem = nn.Sequential(
            nn.Conv3d(in_channels, ch[0], kernel_size=5, stride=2, padding=2, bias=False),
            nn.BatchNorm3d(ch[0]),
            nn.ReLU(inplace=True),
        )

        self.stage1 = nn.Sequential(ResBlock3D(ch[0], ch[1], stride=2, use_cbam=False),
                                    ResBlock3D(ch[1], ch[1], stride=1, use_cbam=False))
        self.stage2 = nn.Sequential(ResBlock3D(ch[1], ch[2], stride=2),
                                    ResBlock3D(ch[2], ch[2], stride=1))
        self.stage3 = nn.Sequential(ResBlock3D(ch[2], ch[3], stride=2),
                                    ResBlock3D(ch[3], ch[3], stride=1))
        self.stage4 = nn.Sequential(ResBlock3D(ch[3], ch[4], stride=2),
                                    ResBlock3D(ch[4], ch[4], stride=1))

        self.pool = nn.AdaptiveAvgPool3d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(ch[4], out_dim),
            nn.LayerNorm(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        return self.head(self.pool(x))


# ── Classifier ────────────────────────────────────────────────────────────────
class LIDCClassifier(nn.Module):
    def __init__(self, num_classes: int = 3):
        super().__init__()
        self.encoder    = Encoder3D(in_channels=C.IN_CHANNELS, out_dim=C.EMBED_DIM)
        self.classifier = nn.Sequential(
            nn.Linear(C.EMBED_DIM, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv3d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm3d, nn.LayerNorm)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def forward(self, image):
        feat = self.encoder(image)
        return self.classifier(feat)           # [B, num_classes] — raw logits

# ==============================================================================
# 8.  FOCAL LOSS
# ==============================================================================
class FocalLoss(nn.Module):
    """
    Multi-class focal loss.
    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)
    alpha is applied as per-class class weights.
    """
    def __init__(self, weight: torch.Tensor, gamma: float = 2.0):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight   # [num_classes]

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_p   = F.log_softmax(logits, dim=1)          # [B, C]
        p       = torch.exp(log_p)                        # [B, C]
        gather  = log_p.gather(1, targets.unsqueeze(1)).squeeze(1)   # [B]
        p_t     = p.gather(1,  targets.unsqueeze(1)).squeeze(1)      # [B]
        focal_w = (1 - p_t) ** self.gamma
        alpha   = self.weight[targets]
        loss    = -alpha * focal_w * gather
        return loss.mean()

# ==============================================================================
# 9.  METRICS HELPER
# ==============================================================================
def compute_metrics(labels: np.ndarray, preds: np.ndarray,
                    probs: np.ndarray, num_classes: int) -> dict:
    acc    = accuracy_score(labels, preds)
    f1_mac = f1_score(labels, preds, average="macro",    zero_division=0)
    f1_wt  = f1_score(labels, preds, average="weighted", zero_division=0)
    prec   = precision_score(labels, preds, average="macro", zero_division=0)
    rec    = recall_score(labels,    preds, average="macro", zero_division=0)

    present = np.unique(labels)
    try:
        auc = roc_auc_score(
            labels, probs[:, :num_classes],
            multi_class="ovr", average="macro",
            labels=list(range(num_classes))
        ) if len(present) > 1 else float("nan")
    except Exception:
        auc = float("nan")

    return dict(acc=acc, f1_mac=f1_mac, f1_wt=f1_wt, prec=prec, rec=rec, auc=auc)

# ==============================================================================
# 10.  TRAINING ENGINE
# ==============================================================================
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion,
    optimizer=None,
    scaler_amp=None,
    training: bool = True,
) -> dict:
    model.train() if training else model.eval()

    losses, all_labels, all_probs = [], [], []

    for batch in loader:
        images  = batch["image"].to(device, non_blocking=True)
        labels  = batch["label"].to(device, non_blocking=True)

        with torch.set_grad_enabled(training):
            with torch.cuda.amp.autocast(enabled=(C.USE_AMP and device.type == "cuda")):
                logits = model(images)
                loss   = criterion(logits, labels)

            if training:
                optimizer.zero_grad(set_to_none=True)
                scaler_amp.scale(loss).backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler_amp.step(optimizer)
                scaler_amp.update()

        probs_cpu  = F.softmax(logits.detach(), dim=1).cpu().numpy()
        labels_cpu = labels.detach().cpu().numpy()

        losses.append(loss.item())
        all_probs.append(probs_cpu)
        all_labels.append(labels_cpu)

    all_probs  = np.vstack(all_probs)
    all_labels = np.concatenate(all_labels).astype(int)
    all_preds  = all_probs.argmax(axis=1)

    metrics = compute_metrics(all_labels, all_preds, all_probs, C.NUM_CLASSES)
    metrics["loss"] = float(np.mean(losses))
    metrics["probs"]  = all_probs
    metrics["labels"] = all_labels
    metrics["preds"]  = all_preds
    return metrics

# ==============================================================================
# 11.  THRESHOLD OPTIMISATION
# ==============================================================================
def tune_thresholds(probs: np.ndarray, labels: np.ndarray,
                    num_classes: int) -> np.ndarray:
    """
    Find per-class probability thresholds maximising macro F1 on a grid search.
    Uses a one-vs-rest approach: for each class, find threshold maximising that
    class's F1 relative to the rest, then pick argmax after threshold scaling.
    """
    thresholds = np.array([0.5] * num_classes)
    best_f1    = 0.0
    grid       = np.arange(0.2, 0.81, 0.05)

    # Exhaustive grid over all class thresholds jointly is 9^3 ≈ 729 combos — fast enough
    import itertools
    for combo in itertools.product(grid, repeat=num_classes):
        t    = np.array(combo)
        # scale probs by thresholds then argmax
        scaled = probs / (t + 1e-9)
        preds  = scaled.argmax(axis=1)
        f1     = f1_score(labels, preds, average="macro", zero_division=0)
        if f1 > best_f1:
            best_f1    = f1
            thresholds = t

    log.info(f"  Tuned thresholds: {thresholds.tolist()}  → macro F1={best_f1:.4f}")
    return thresholds


def apply_thresholds(probs: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
    scaled = probs / (thresholds + 1e-9)
    return scaled.argmax(axis=1)

# ==============================================================================
# 12.  PLOTTING HELPERS
# ==============================================================================
def _save(fig, name):
    p = os.path.join(C.OUTPUT_DIR, name)
    fig.savefig(p, bbox_inches="tight", dpi=150)
    plt.close(fig)
    log.info(f"  Saved plot → {p}")


def plot_confusion_matrix(labels, preds, fold: int = None):
    cm  = confusion_matrix(labels, preds, labels=list(range(C.NUM_CLASSES)))
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=C.CLASS_NAMES, yticklabels=C.CLASS_NAMES, ax=ax)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    title = f"Confusion Matrix" + (f" — Fold {fold}" if fold else "")
    ax.set_title(title)
    _save(fig, f"confusion_matrix{'_fold' + str(fold) if fold else ''}.png")


def plot_training_curves(history: list, fold: int = None):
    epochs = [h["epoch"] for h in history]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key, label in zip(
        axes,
        ["loss",    "f1_mac",   "auc"],
        ["Loss",    "Macro F1", "ROC-AUC"]
    ):
        tr = [h[f"train_{key}"] for h in history]
        vl = [h[f"val_{key}"]   for h in history]
        ax.plot(epochs, tr, label="Train")
        ax.plot(epochs, vl, label="Val")
        ax.set_title(label); ax.set_xlabel("Epoch")
        ax.legend()
    fig.tight_layout()
    _save(fig, f"training_curves{'_fold' + str(fold) if fold else ''}.png")


def plot_calibration(probs: np.ndarray, labels: np.ndarray, fold: int = None):
    fig, ax = plt.subplots(figsize=(6, 5))
    for cls in range(C.NUM_CLASSES):
        binary_labels = (labels == cls).astype(int)
        cls_probs     = probs[:, cls]
        if binary_labels.sum() < 2:
            continue
        fraction_pos, mean_pred = calibration_curve(binary_labels, cls_probs, n_bins=10)
        ax.plot(mean_pred, fraction_pos, marker="o", label=C.CLASS_NAMES[cls])
    ax.plot([0, 1], [0, 1], "k--", label="Perfect")
    ax.set_xlabel("Mean predicted probability")
    ax.set_ylabel("Fraction of positives")
    ax.set_title("Calibration Curves" + (f" — Fold {fold}" if fold else ""))
    ax.legend()
    _save(fig, f"calibration{'_fold' + str(fold) if fold else ''}.png")

# ==============================================================================
# 13.  SINGLE FOLD TRAINING LOOP
# ==============================================================================
def train_fold(
    train_df: pd.DataFrame,
    val_df:   pd.DataFrame,
    fold:     int,
    class_weights: np.ndarray,
) -> dict:
    log.info(f"\n{'='*60}")
    log.info(f"  FOLD {fold}  |  train={len(train_df)}  val={len(val_df)}")
    log.info(f"{'='*60}")

    train_ds = LIDCDataset(train_df, training=True)
    val_ds   = LIDCDataset(val_df,   training=False)

    train_ld = DataLoader(train_ds, batch_size=C.BATCH_SIZE, shuffle=True,
                          num_workers=C.NUM_WORKERS, pin_memory=(device.type == "cuda"))
    val_ld   = DataLoader(val_ds,   batch_size=C.BATCH_SIZE, shuffle=False,
                          num_workers=C.NUM_WORKERS, pin_memory=(device.type == "cuda"))

    model = LIDCClassifier(num_classes=C.NUM_CLASSES).to(device)

    cw_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

    if C.USE_FOCAL:
        criterion = FocalLoss(weight=cw_tensor, gamma=C.FOCAL_GAMMA)
    else:
        criterion = nn.CrossEntropyLoss(weight=cw_tensor)

    optimizer   = torch.optim.AdamW(model.parameters(), lr=C.LR, weight_decay=C.WEIGHT_DECAY)
    scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=C.EPOCHS, eta_min=C.LR_MIN
    )
    scaler_amp  = torch.cuda.amp.GradScaler(enabled=(C.USE_AMP and device.type == "cuda"))

    best_score   = -1.0
    best_state   = None
    patience_cnt = 0
    history      = []

    for epoch in range(1, C.EPOCHS + 1):
        t_start = time.time()

        tr = run_epoch(model, train_ld, criterion, optimizer, scaler_amp, training=True)
        vl = run_epoch(model, val_ld,   criterion, training=False)
        scheduler.step()

        score = vl["f1_mac"] if math.isnan(vl["auc"]) else vl["auc"]

        history.append({
            "epoch": epoch,
            **{f"train_{k}": tr[k] for k in ["loss", "f1_mac", "auc"]},
            **{f"val_{k}":   vl[k] for k in ["loss", "f1_mac", "auc"]},
        })

        elapsed = time.time() - t_start
        log.info(
            f"  Epoch {epoch:02d}/{C.EPOCHS}  "
            f"| TrLoss {tr['loss']:.4f} TrF1 {tr['f1_mac']:.4f} TrAUC {tr['auc']:.4f}  "
            f"| VlLoss {vl['loss']:.4f} VlF1 {vl['f1_mac']:.4f} VlAUC {vl['auc']:.4f}  "
            f"| {elapsed:.1f}s"
        )

        if score > best_score:
            best_score   = score
            best_state   = deepcopy(model.state_dict())
            patience_cnt = 0
            path = os.path.join(C.OUTPUT_DIR, f"{C.MODEL_NAME}_fold{fold}.pt")
            torch.save({
                "model_state_dict": best_state,
                "num_classes": C.NUM_CLASSES,
                "in_channels": C.IN_CHANNELS,
                "target_spacing": C.TARGET_SPACING,
                "target_shape":   C.TARGET_SHAPE,
                "hu_min": C.HU_MIN, "hu_max": C.HU_MAX,
                "hu_lung_min": C.HU_LUNG_MIN, "hu_lung_max": C.HU_LUNG_MAX,
                "use_dual_channel": C.USE_DUAL_CHANNEL,
                "class_names": C.CLASS_NAMES,
                "fold": fold,
            }, path)
            log.info(f"    ✓ New best (score={best_score:.4f}) saved → {path}")
        else:
            patience_cnt += 1
            if patience_cnt >= C.PATIENCE:
                log.info(f"    Early stopping at epoch {epoch}")
                break

    # Reload best weights and evaluate
    model.load_state_dict(best_state)
    vl_final = run_epoch(model, val_ld, criterion, training=False)

    # Threshold tuning
    thresholds = np.array([0.5] * C.NUM_CLASSES)
    if C.TUNE_THRESHOLDS:
        thresholds = tune_thresholds(vl_final["probs"], vl_final["labels"], C.NUM_CLASSES)

    preds_tuned = apply_thresholds(vl_final["probs"], thresholds)
    metrics_tuned = compute_metrics(
        vl_final["labels"], preds_tuned, vl_final["probs"], C.NUM_CLASSES
    )

    log.info(f"\n  ── Fold {fold} Best Validation ──")
    log.info(f"  Accuracy:   {metrics_tuned['acc']:.4f}")
    log.info(f"  Macro F1:   {metrics_tuned['f1_mac']:.4f}")
    log.info(f"  Weighted F1:{metrics_tuned['f1_wt']:.4f}")
    log.info(f"  ROC-AUC:    {metrics_tuned['auc']:.4f}" if not math.isnan(metrics_tuned["auc"]) else "  ROC-AUC: N/A")
    log.info("\n" + classification_report(
        vl_final["labels"], preds_tuned,
        target_names=C.CLASS_NAMES, digits=4, zero_division=0
    ))

    plot_confusion_matrix(vl_final["labels"], preds_tuned, fold=fold)
    plot_training_curves(history, fold=fold)
    plot_calibration(vl_final["probs"], vl_final["labels"], fold=fold)

    return {
        "fold":       fold,
        "thresholds": thresholds.tolist(),
        "metrics":    metrics_tuned,
        "history":    history,
        "probs":      vl_final["probs"],
        "labels":     vl_final["labels"],
        "preds":      preds_tuned,
        "model":      model,
    }

# ==============================================================================
# 14.  MAIN — DATA LOADING, K-FOLD, AGGREGATE RESULTS
# ==============================================================================
def main():
    # ── Load Excel ────────────────────────────────────────────────────────
    log.info("Reading Excel …")
    ext = os.path.splitext(C.EXCEL_PATH)[-1].lower()
    if ext == ".csv":
        df_raw = pd.read_csv(C.EXCEL_PATH)
    elif ext == ".xls":
        df_raw = pd.read_excel(C.EXCEL_PATH, engine="xlrd")
    else:
        df_raw = pd.read_excel(C.EXCEL_PATH)

    assert C.PATIENT_ID_COL in df_raw.columns,   f"Missing column: {C.PATIENT_ID_COL}"
    assert C.LABEL_SOURCE_COL in df_raw.columns, f"Missing column: {C.LABEL_SOURCE_COL}"

    df_raw["patient_id"] = df_raw[C.PATIENT_ID_COL].astype(str).map(normalize_patient_id)
    df_raw["label"]      = df_raw[C.LABEL_SOURCE_COL].map(
        lambda x: C.LABEL_MAP.get(int(x) if not pd.isna(x) else 0, 0)
    )

    # ── DICOM discovery ───────────────────────────────────────────────────
    patient_to_series = build_patient_to_series_map(C.DICOM_ROOT)
    df_raw["dicom_dir"] = df_raw["patient_id"].map(patient_to_series)

    before = len(df_raw)
    df     = df_raw.dropna(subset=["dicom_dir", "label"]).copy()
    df["label"] = df["label"].astype(int)
    log.info(f"Matched {len(df)}/{before} patients with DICOM scans.")

    if len(df) < 10:
        raise RuntimeError("Fewer than 10 matched rows — check DICOM_ROOT paths.")

    log.info("\nClass distribution:")
    for cls, name in enumerate(C.CLASS_NAMES):
        n = (df["label"] == cls).sum()
        log.info(f"  {cls} — {name}: {n}")

    # ── Class weights (inverse frequency) ─────────────────────────────────
    counts        = np.array([(df["label"] == c).sum() for c in range(C.NUM_CLASSES)], dtype=float)
    class_weights = counts.sum() / (C.NUM_CLASSES * np.maximum(counts, 1))
    class_weights = class_weights / class_weights.sum() * C.NUM_CLASSES  # normalise so sum = num_classes
    log.info(f"Class weights: {class_weights.tolist()}")

    # ── K-Fold or single split ─────────────────────────────────────────────
    labels_array = df["label"].values
    fold_results = []

    if C.N_FOLDS > 1:
        skf = StratifiedKFold(n_splits=C.N_FOLDS, shuffle=True, random_state=C.SEED)
        splits = list(skf.split(df, labels_array))
    else:
        from sklearn.model_selection import train_test_split
        tr_idx, vl_idx = train_test_split(
            np.arange(len(df)), test_size=C.VAL_SIZE,
            stratify=labels_array, random_state=C.SEED
        )
        splits = [(tr_idx, vl_idx)]

    for fold, (tr_idx, vl_idx) in enumerate(splits, start=1):
        result = train_fold(
            df.iloc[tr_idx].copy(),
            df.iloc[vl_idx].copy(),
            fold          = fold,
            class_weights = class_weights,
        )
        fold_results.append(result)

    # ── Aggregate results across folds ────────────────────────────────────
    log.info("\n" + "=" * 60)
    log.info("CROSS-VALIDATION SUMMARY")
    log.info("=" * 60)

    keys = ["acc", "f1_mac", "f1_wt", "auc"]
    agg  = {k: [r["metrics"][k] for r in fold_results] for k in keys}

    for k in keys:
        vals = [v for v in agg[k] if not math.isnan(v)]
        if vals:
            log.info(f"  {k.upper():10s}: {np.mean(vals):.4f} ± {np.std(vals):.4f}")

    # Pool all OOF predictions for a global confusion matrix & calibration
    all_labels = np.concatenate([r["labels"] for r in fold_results])
    all_preds  = np.concatenate([r["preds"]  for r in fold_results])
    all_probs  = np.vstack(    [r["probs"]   for r in fold_results])

    log.info("\nOverall OOF classification report:")
    log.info(classification_report(all_labels, all_preds,
             target_names=C.CLASS_NAMES, digits=4, zero_division=0))

    plot_confusion_matrix(all_labels, all_preds)    # overall (no fold suffix)
    plot_calibration(all_probs, all_labels)

    # Save summary JSON
    summary = {
        "cv_means": {k: float(np.nanmean(agg[k])) for k in keys},
        "cv_stds":  {k: float(np.nanstd(agg[k]))  for k in keys},
        "folds": [
            {"fold": r["fold"], "thresholds": r["thresholds"],
             "metrics": {k: r["metrics"][k] for k in keys}}
            for r in fold_results
        ],
    }
    summary_path = os.path.join(C.OUTPUT_DIR, "cv_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    log.info(f"\nSaved CV summary → {summary_path}")
    log.info("Done.")

    return fold_results


if __name__ == "__main__":
    main()

2026-06-10 19:44:23,036  Device: cuda
2026-06-10 19:44:23,042  Reading Excel …
2026-06-10 19:44:23,065  Scanning DICOM directory tree …
2026-06-10 19:44:35,282    Total series directories found: 1246
2026-06-10 19:44:35,731    Mapped 948 unique patients.
2026-06-10 19:44:35,751  Matched 151/157 patients with DICOM scans.
2026-06-10 19:44:35,751  
Class distribution:
2026-06-10 19:44:35,752    0 — Benign/Unknown: 62
2026-06-10 19:44:35,753    1 — Primary Lung Ca: 39
2026-06-10 19:44:35,753    2 — Metastatic: 50
2026-06-10 19:44:35,754  Class weights: [0.7833422603106589, 1.2453133369041245, 0.9713444027852169]
2026-06-10 19:44:35,758  
2026-06-10 19:44:35,758    FOLD 1  |  train=120  val=31
2026-06-10 19:44:35,759  ============================================================
2026-06-10 19:58:39,244    Epoch 01/40  | TrLoss 0.8464 TrF1 0.3057 TrAUC nan  | VlLoss 0.4770 VlF1 0.1368 VlAUC 0.5741  | 843.4s
2026-06-10 19:58:40,450      ✓ New best (score=0.5741) saved → lidc_output\best_lidc_

In [22]:
# ==============================================================================
# LIDC-IDRI  |  Multiclass Model — Test & Inference Script
# ==============================================================================
# Loads one or more fold checkpoints saved by lidc_multiclass_pipeline.py,
# randomly samples N patients from the full dataset, runs predictions, and
# prints a detailed per-patient comparison vs ground truth.
#
# Usage:
#   python lidc_multiclass_test.py
# ==============================================================================

import os, re, math, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pydicom
from scipy.ndimage import zoom, gaussian_filter, map_coordinates

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# ==============================================================================
# CONFIGURATION  — must match training config
# ==============================================================================
EXCEL_PATH   = r"../LIDC-IDRI-133GB/Data/tcia-diagnosis-data-2012-04-20.xls"
DICOM_ROOT   = r"../LIDC-IDRI-133GB/Data/manifest-1600709154662/LIDC-IDRI"
OUTPUT_DIR   = "lidc_output"        # where fold checkpoints were saved
MODEL_NAME   = "best_lidc_model"    # prefix; will auto-detect fold files

PATIENT_ID_COL   = "TCIA Patient ID"
LABEL_SOURCE_COL = (
    "Diagnosis at the Patient Level\n"
    "0=Unknown\n"
    "1=benign or non-malignant disease\n"
    "2= malignant, primary lung cancer\n"
    "3 = malignant metastatic\n"
)
LABEL_MAP   = {0: 0, 1: 0, 2: 1, 3: 2}
CLASS_NAMES = ["Benign/Unknown", "Primary Lung Ca", "Metastatic"]
NUM_CLASSES = 3

N_SAMPLES   = 20     # patients to randomly draw; None = all matched patients
SEED        = 777    # independent from training seed
USE_AMP     = True

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ==============================================================================
# HELPERS  (identical to pipeline)
# ==============================================================================
def normalize_patient_id(x):
    s = str(x).strip().replace("\\", "/")
    s = os.path.basename(s)
    s = re.sub(r"\.(xml|xlsx?)$", "", s, flags=re.IGNORECASE)

    m = re.search(r"LIDC-IDRI-\d+", s, flags=re.IGNORECASE)
    if m:
        num = int(re.search(r"\d+", m.group(0)).group(0))
        return f"LIDC-IDRI-{num:04d}"

    if re.fullmatch(r"\d+", s):
        return f"LIDC-IDRI-{int(s):04d}"

    return s

def build_patient_to_series_map(root_dir):
    patient_map = {}
    for cur, dirs, files in os.walk(root_dir):
        dcm_files = [f for f in files if f.lower().endswith(".dcm")]
        if not dcm_files:
            continue
        # find ALL patient-ID segments in path, take the deepest one
        all_matches = re.findall(r"LIDC-IDRI-\d+", cur, flags=re.IGNORECASE)
        if not all_matches:
            continue
        pid = normalize_patient_id(all_matches[-1])  # last = deepest in path
        n   = len(dcm_files)
        if pid not in patient_map or n > patient_map[pid][1]:
            patient_map[pid] = (cur, n)
    return {k: v[0] for k, v in patient_map.items()}

def load_dicom_series(dicom_dir):
    dicom_dir = Path(dicom_dir)
    files = sorted(dicom_dir.glob("*.dcm")) or sorted(dicom_dir.iterdir())
    slices = []
    for fp in files:
        try:
            ds = pydicom.dcmread(str(fp), force=True)
            if hasattr(ds, "PixelData"):
                slices.append(ds)
        except Exception:
            pass
    if not slices:
        raise ValueError(f"No readable DICOM slices in {dicom_dir}")
    def sk(ds):
        try:    return float(ds.ImagePositionPatient[2])
        except Exception: pass
        try:    return int(ds.InstanceNumber)
        except Exception: return 0
    slices.sort(key=sk)
    volume = np.empty((len(slices), slices[0].Rows, slices[0].Columns), dtype=np.float32)

    for i, ds in enumerate(slices):

        arr = ds.pixel_array.astype(np.float32)

        slope = float(getattr(ds, 'RescaleSlope', 1.0))

        inter = float(getattr(ds, 'RescaleIntercept', 0.0))

        volume[i] = arr * slope + inter

        if hasattr(ds, '_pixel_array'): del ds._pixel_array

        if hasattr(ds, 'PixelData'): del ds.PixelData
    try:
        zs   = [float(ds.ImagePositionPatient[2]) for ds in slices]
        z_sp = abs(zs[1] - zs[0]) if len(zs) > 1 else float(getattr(slices[0], "SliceThickness", 2.0))
        z_sp = z_sp or float(getattr(slices[0], "SliceThickness", 2.0))
    except Exception:
        z_sp = float(getattr(slices[0], "SliceThickness", 2.0))
    ps = getattr(slices[0], "PixelSpacing", [1.0, 1.0])
    return volume, (z_sp, float(ps[0]), float(ps[1]))

def clip_norm(vol, hu_min, hu_max):
    v = np.clip(vol, hu_min, hu_max)
    return ((v - hu_min) / (hu_max - hu_min + 1e-8)).astype(np.float32)

def resample(vol, spacing, target_spacing):
    f = tuple(s / t for s, t in zip(spacing, target_spacing))
    import torch
    import torch.nn.functional as F
    new_shape = (
        int(round(vol.shape[0] * f[0])),
        int(round(vol.shape[1] * f[1])),
        int(round(vol.shape[2] * f[2]))
    )
    vol_t = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0)
    vol_t = F.interpolate(vol_t, size=new_shape, mode='trilinear', align_corners=False)
    return vol_t.squeeze(0).squeeze(0).numpy()

def crop_pad(vol, shape):
    tz, ty, tx = shape; z, y, x = vol.shape
    out = np.zeros(shape, dtype=vol.dtype)
    z0s = max((z-tz)//2,0); y0s = max((y-ty)//2,0); x0s = max((x-tx)//2,0)
    z1s = min(z0s+tz,z);    y1s = min(y0s+ty,y);    x1s = min(x0s+tx,x)
    cz,cy,cx = z1s-z0s, y1s-y0s, x1s-x0s
    z0d=max((tz-cz)//2,0); y0d=max((ty-cy)//2,0); x0d=max((tx-cx)//2,0)
    out[z0d:z0d+cz, y0d:y0d+cy, x0d:x0d+cx] = vol[z0s:z1s, y0s:y1s, x0s:x1s]
    return out

def preprocess_ct(dicom_dir, ckpt_meta):
    target_spacing   = tuple(ckpt_meta["target_spacing"])
    target_shape     = tuple(ckpt_meta["target_shape"])
    use_dual_channel = ckpt_meta["use_dual_channel"]

    volume, spacing = load_dicom_series(dicom_dir)
    volume = resample(volume, spacing, target_spacing)
    volume = crop_pad(volume, target_shape)

    ch_std  = clip_norm(volume, ckpt_meta["hu_min"],      ckpt_meta["hu_max"])
    if use_dual_channel:
        ch_lung = clip_norm(volume, ckpt_meta["hu_lung_min"], ckpt_meta["hu_lung_max"])
        return np.stack([ch_std, ch_lung], axis=0)
    return ch_std[np.newaxis]

# ==============================================================================
# MODEL  (must mirror pipeline exactly)
# ==============================================================================
class SEBlock3D(nn.Module):
    def __init__(self, channels, reduction=8):
        super().__init__()
        h = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.fc   = nn.Sequential(nn.Linear(channels, h), nn.ReLU(True),
                                   nn.Linear(h, channels), nn.Sigmoid())
    def forward(self, x):
        b, c = x.shape[:2]
        return x * self.fc(self.pool(x).view(b, c)).view(b, c, 1, 1, 1)

class CBAMSpatial3D(nn.Module):
    def __init__(self, k=7):
        super().__init__()
        self.conv = nn.Conv3d(2, 1, k, padding=k // 2, bias=False)

    def forward(self, x):
        avg_out = x.mean(1, keepdim=True)
        max_out = x.max(1, keepdim=True).values
        w = torch.sigmoid(self.conv(torch.cat([avg_out, max_out], dim=1)))
        return x * w


class CBAM3D(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.se = SEBlock3D(ch)
        self.spatial = CBAMSpatial3D()

    def forward(self, x):
        return self.spatial(self.se(x))


class ResBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1, use_cbam=True):
        super().__init__()

        self.conv1 = nn.Conv3d(
            in_ch, out_ch,
            kernel_size=3,
            stride=stride,
            padding=1,
            bias=False
        )
        self.bn1 = nn.BatchNorm3d(out_ch)

        self.conv2 = nn.Conv3d(
            out_ch, out_ch,
            kernel_size=3,
            padding=1,
            bias=False
        )
        self.bn2 = nn.BatchNorm3d(out_ch)

        self.attn = CBAM3D(out_ch) if use_cbam else nn.Identity()

        self.down = (
            nn.Sequential(
                nn.Conv3d(
                    in_ch,
                    out_ch,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),
                nn.BatchNorm3d(out_ch)
            )
            if stride != 1 or in_ch != out_ch
            else nn.Identity()
        )

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.down(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out = self.attn(out)

        out += identity
        out = self.relu(out)

        return out


class Encoder3D(nn.Module):

    CHANNELS = [16, 32, 64, 128, 256]

    def __init__(self, in_ch=1, out_dim=256):
        super().__init__()

        ch = self.CHANNELS

        self.stem = nn.Sequential(
            nn.Conv3d(
                in_ch,
                ch[0],
                kernel_size=5,
                stride=2,
                padding=2,
                bias=False
            ),
            nn.BatchNorm3d(ch[0]),
            nn.ReLU(inplace=True)
        )

        self.stage1 = nn.Sequential(
            ResBlock3D(ch[0], ch[1], stride=2, use_cbam=False),
            ResBlock3D(ch[1], ch[1], stride=1, use_cbam=False)
        )

        self.stage2 = nn.Sequential(
            ResBlock3D(ch[1], ch[2], stride=2),
            ResBlock3D(ch[2], ch[2])
        )

        self.stage3 = nn.Sequential(
            ResBlock3D(ch[2], ch[3], stride=2),
            ResBlock3D(ch[3], ch[3])
        )

        self.stage4 = nn.Sequential(
            ResBlock3D(ch[3], ch[4], stride=2),
            ResBlock3D(ch[4], ch[4])
        )

        self.pool = nn.AdaptiveAvgPool3d(1)

        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(ch[4], out_dim),
            nn.LayerNorm(out_dim),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3)
        )

    def forward(self, x):

        x = self.stem(x)

        x = self.stage1(x)
        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)

        x = self.pool(x)

        return self.head(x)

# ==============================================================================
# LOAD ALL FOLD CHECKPOINTS  (ensemble by averaging probabilities)
# ==============================================================================
print(f"\nLooking for fold checkpoints in: {OUTPUT_DIR}")
ckpt_paths = sorted(Path(OUTPUT_DIR).glob(f"{MODEL_NAME}_fold*.pt"))
if not ckpt_paths:
    raise FileNotFoundError(
        f"No checkpoint files found matching '{MODEL_NAME}_fold*.pt' in {OUTPUT_DIR}. "
        "Run the training pipeline first."
    )

models_meta = []
for path in ckpt_paths:
    ckpt = torch.load(str(path), map_location=device)
    meta = {k: ckpt[k] for k in ["num_classes","in_channels","target_spacing","target_shape",
                                   "hu_min","hu_max","hu_lung_min","hu_lung_max",
                                   "use_dual_channel","class_names"]}
    m = LIDCClassifier(
        num_classes  = meta["num_classes"],
        in_channels  = meta["in_channels"],
        embed_dim    = 256,
    ).to(device)
    m.load_state_dict(ckpt["model_state_dict"])
    m.eval()
    models_meta.append((m, meta))
    print(f"  Loaded fold {ckpt.get('fold','?')} → {path.name}")

ckpt_meta = models_meta[0][1]   # use first for preprocessing params (same across folds)
print(f"Ensemble of {len(models_meta)} model(s) ready.\n")

# ==============================================================================
# BUILD DATA TABLE
# ==============================================================================
print("Scanning DICOM folders …")
patient_to_series = build_patient_to_series_map(DICOM_ROOT)
print(f"Discovered {len(patient_to_series)} patients with DICOM data.\n")

print("Reading Excel …")
ext = os.path.splitext(EXCEL_PATH)[-1].lower()
df_raw = pd.read_excel(EXCEL_PATH, engine="xlrd") if ext == ".xls" else pd.read_excel(EXCEL_PATH)
df_raw["patient_id"] = df_raw[PATIENT_ID_COL].astype(str).map(normalize_patient_id)
df_raw["label"]      = df_raw[LABEL_SOURCE_COL].map(
    lambda x: LABEL_MAP.get(int(x) if not pd.isna(x) else 0, 0)
)
df_raw["dicom_dir"]  = df_raw["patient_id"].map(patient_to_series)
df = df_raw.dropna(subset=["dicom_dir","label"]).copy()
df["label"] = df["label"].astype(int)
print(f"Total matched patients: {len(df)}")

# Class distribution
for cls, name in enumerate(CLASS_NAMES):
    n = (df["label"] == cls).sum()
    print(f"  Class {cls} — {name}: {n}")

# ==============================================================================
# RANDOM SAMPLING
# ==============================================================================
n = min(N_SAMPLES, len(df)) if N_SAMPLES else len(df)
sample = df.sample(n=n, random_state=SEED).reset_index(drop=True)
print(f"\nRandomly selected {n} patients for testing (seed={SEED})")

# ==============================================================================
# ENSEMBLE INFERENCE
# ==============================================================================
def ensemble_predict(dicom_dir):
    vol = preprocess_ct(dicom_dir, ckpt_meta)
    img = torch.tensor(vol, dtype=torch.float32).unsqueeze(0).to(device)
    prob_sum = None
    with torch.no_grad():
        for model, _ in models_meta:
            with torch.cuda.amp.autocast(enabled=(USE_AMP and device.type == "cuda")):
                logits = model(img)
            p = F.softmax(logits, dim=1).cpu().numpy()[0]
            prob_sum = p if prob_sum is None else prob_sum + p
    probs = prob_sum / len(models_meta)
    return probs   # [NUM_CLASSES]

# ==============================================================================
# RUN TEST
# ==============================================================================
col_w = max(len(n) for n in CLASS_NAMES) + 2
header = (f"{'#':<4} {'Patient ID':<22} {'Actual':<{col_w}} "
          f"{'Predicted':<{col_w}} "
          + "  ".join(f"P({n})" for n in CLASS_NAMES)
          + "  Correct?")
print("\n" + header)
print("-" * len(header))

results = []
for i, row in sample.iterrows():
    pid    = row["patient_id"]
    actual = int(row["label"])

    try:
        probs     = ensemble_predict(row["dicom_dir"])
        predicted = int(probs.argmax())
        correct   = "✓" if predicted == actual else "✗"
        err       = None
    except Exception as e:
        probs     = np.full(NUM_CLASSES, float("nan"))
        predicted = -1
        correct   = "ERR"
        err       = str(e)

    results.append(dict(
        patient_id       = pid,
        actual           = actual,
        actual_label     = CLASS_NAMES[actual] if actual < NUM_CLASSES else "?",
        predicted        = predicted,
        predicted_label  = CLASS_NAMES[predicted] if 0 <= predicted < NUM_CLASSES else "Error",
        correct          = correct == "✓",
        error            = err,
        **{f"prob_{CLASS_NAMES[c]}": probs[c] for c in range(NUM_CLASSES)},
    ))

    prob_str = "  ".join(f"{probs[c]:6.4f}" for c in range(NUM_CLASSES))
    print(f"{len(results):<4} {pid:<22} {CLASS_NAMES[actual]:<{col_w}} "
          f"{CLASS_NAMES[predicted] if predicted >= 0 else 'Error':<{col_w}} "
          f"{prob_str}  {correct}")

# ==============================================================================
# AGGREGATE METRICS
# ==============================================================================
results_df = pd.DataFrame(results)
valid      = results_df[results_df["predicted"] >= 0].copy()

print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)
print(f"Total tested:        {len(results_df)}")
print(f"Successful:          {len(valid)}")
print(f"Errors:              {len(results_df) - len(valid)}")

if len(valid) > 0:
    y_true  = valid["actual"].values.astype(int)
    y_pred  = valid["predicted"].values.astype(int)
    y_probs = valid[[f"prob_{n}" for n in CLASS_NAMES]].values

    acc    = accuracy_score(y_true, y_pred)
    f1_mac = f1_score(y_true, y_pred, average="macro",    zero_division=0)
    f1_wt  = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    try:
        auc = roc_auc_score(y_true, y_probs, multi_class="ovr",
                            average="macro", labels=list(range(NUM_CLASSES)))
    except Exception:
        auc = float("nan")

    print(f"\nAccuracy:           {acc:.4f}")
    print(f"Macro F1:           {f1_mac:.4f}")
    print(f"Weighted F1:        {f1_wt:.4f}")
    print(f"ROC-AUC (OvR):     {auc:.4f}" if not math.isnan(auc) else "ROC-AUC:            N/A (need ≥2 classes)")

    print("\nConfusion Matrix (rows=Actual, cols=Predicted):")
    cm = confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES)))
    header_cm = "            " + "  ".join(f"{n[:10]:>10}" for n in CLASS_NAMES)
    print(header_cm)
    for r_idx, name in enumerate(CLASS_NAMES):
        row_str = "  ".join(f"{cm[r_idx, c_idx]:>10}" for c_idx in range(NUM_CLASSES))
        print(f"{name[:12]:12} {row_str}")

    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                digits=4, zero_division=0))

# Save
out_path = os.path.join(OUTPUT_DIR, "test_predictions_multiclass.csv")
results_df.to_csv(out_path, index=False)
print(f"Results saved → {out_path}")

# Print errors
errs = results_df[results_df["error"].notna()]
if len(errs):
    print(f"\n  {len(errs)} inference error(s):")
    for _, r in errs.iterrows():
        print(f"  {r['patient_id']}: {r['error']}")

Device: cuda

Looking for fold checkpoints in: lidc_output
  Loaded fold 1 → best_lidc_model_fold1.pt
  Loaded fold 2 → best_lidc_model_fold2.pt
  Loaded fold 3 → best_lidc_model_fold3.pt
  Loaded fold 4 → best_lidc_model_fold4.pt
  Loaded fold 5 → best_lidc_model_fold5.pt
Ensemble of 5 model(s) ready.

Scanning DICOM folders …
Discovered 948 patients with DICOM data.

Reading Excel …
Total matched patients: 151
  Class 0 — Benign/Unknown: 62
  Class 1 — Primary Lung Ca: 39
  Class 2 — Metastatic: 50

Randomly selected 20 patients for testing (seed=777)

#    Patient ID             Actual            Predicted         P(Benign/Unknown)  P(Primary Lung Ca)  P(Metastatic)  Correct?
------------------------------------------------------------------------------------------------------------------------------
1    LIDC-IDRI-0289         Benign/Unknown    Benign/Unknown    0.3677  0.3240  0.3083  ✓
2    LIDC-IDRI-0138         Primary Lung Ca   Benign/Unknown    0.3909  0.2761  0.3328  ✗
3    

In [ ]:
# pip cache purge

Files removed: 1801 (1994.4 MB)
Directories removed: 691
Note: you may need to restart the kernel to use updated packages.
